In [1]:
# inlegalbert_bilstm_mha_crf_discourse_smote.py
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# KEY ADDITION vs baseline (v2):
#   ╔══════════════════════════════════════════════════════════════════╗
#   ║         DISCOURSE-AWARE SMOTE  (DA-SMOTE)                       ║
#   ║                                                                  ║
#   ║  Motivation: Standard SMOTE treats every sentence as i.i.d.,    ║
#   ║  ignoring the sequential/discourse structure of legal documents. ║
#   ║  Interpolating a FAC sentence near a RATIO sentence would        ║
#   ║  produce an embedding that belongs to neither class and          ║
#   ║  injects noise.                                                  ║
#   ║                                                                  ║
#   ║  DA-SMOTE enforces THREE discourse constraints:                  ║
#   ║  1. Intra-class neighbourhood  – only sentences of the SAME      ║
#   ║     rhetorical role are eligible as SMOTE neighbours.            ║
#   ║  2. Positional coherence       – synthetic neighbour is drawn    ║
#   ║     from the k nearest same-class sentences whose normalised     ║
#   ║     document position is within ±POSITION_WINDOW of the seed,   ║
#   ║     respecting the flow PREAMBLE→FAC→...→RPC.                   ║
#   ║  3. Transition-aware insertion – synthetic sentences are grouped ║
#   ║     into pseudo-documents that honour observed label bigram       ║
#   ║     transitions (Markov chain re-assembly), so the context       ║
#   ║     BiLSTM sees plausible discourse sequences.                   ║
#   ║                                                                  ║
#   ║  Two interpolation modes:                                        ║
#   ║   • "embedding"  – λ·e_i + (1-λ)·e_j  (classic SMOTE in        ║
#   ║                    BERT embedding space)                         ║
#   ║   • "manifold"   – interpolation along the PCA manifold of       ║
#   ║                    the class cluster to stay on the data         ║
#   ║                    manifold.                                     ║
#   ╚══════════════════════════════════════════════════════════════════╝
#
# ALL anti-overfitting settings from v2 are retained:
#   early stopping, layer freeze, LR decay, dropout=0.4, wd=0.05,
#   aux CE loss, reduced LSTM sizes.
#

import os, json, random, time
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_dasmote_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# ── BERT freeze / layer-wise LR decay ─────────────────────
BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

# ── Sentence-level BiLSTM ──────────────────────────────────
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

# ── Context BiLSTM ────────────────────────────────────────
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

# ── Auxiliary loss ─────────────────────────────────────────
AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# ── Early stopping ─────────────────────────────────────────
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD  = 0.05

# ══════════════════════════════════════════════════════════════
# DISCOURSE-AWARE SMOTE CONFIG
# ══════════════════════════════════════════════════════════════
DA_SMOTE_ENABLED         = True    # master switch

# Neighbourhood
DA_K_NEIGHBORS           = 5       # k nearest same-class neighbours
DA_POSITION_WINDOW       = 0.15    # ±15 % of doc length for positional filter

# Interpolation
DA_INTERP_MODE           = "manifold"   # "embedding" | "manifold"
DA_PCA_COMPONENTS        = 32      # PCA dims for manifold mode (≤ min class size)
DA_LAMBDA_LOW            = 0.3     # λ ∈ [DA_LAMBDA_LOW, DA_LAMBDA_HIGH]
DA_LAMBDA_HIGH           = 0.7

# Oversampling target
# For each rare class, generate enough synthetics so it reaches
# DA_TARGET_RATIO * majority_class_count sentences.
DA_TARGET_RATIO          = 0.60    # 60 % of majority count
DA_MIN_CLASS_SIZE        = 6       # skip a class if it has fewer real samples

# Pseudo-doc assembly
DA_PSEUDO_DOC_SIZE       = 32      # sentences per synthetic pseudo-document
DA_SYNTHETIC_WEIGHT      = 0.4     # loss weight for synthetic pseudo-doc batches
DA_USE_TRANSITION_CHAIN  = True    # Markov-chain re-assembly of pseudo-docs

# Embedding extraction
DA_EMBED_BATCH_SIZE      = 64

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":          "InLegalBERT Encoder",
        "sent_bilstm":   "Sentence BiLSTM",
        "mha_pooling":   "Multi-Head Attn Pooling",
        "ctx_bilstm":    "Context BiLSTM",
        "classifier":    "Classifier Head",
        "crf":           "CRF",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({"Component": name, "Trainable Params": trainable,
                     "Frozen Params": frozen, "Total Params": trainable + frozen})

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({"Component": "── TOTAL ──", "Trainable Params": total_trainable,
                 "Frozen Params": total_frozen, "Total Params": total_trainable + total_frozen})

    print("\n" + "=" * 72)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT+BiLSTM+MHA+CRF  DA-SMOTE)")
    print("=" * 72)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 72)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 72)
        print(f"  {r['Component']:<30} {r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} {r['Total Params']:>12,}")
    print("=" * 72)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids     = [lid for _, labs in docs for lid in labs]
    total       = len(all_ids)
    counts      = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET  – original tokenised docs
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                       torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L, B  = batch[0]["input_ids"].shape[1], len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# DATASET  – pre-computed embedding pseudo-docs (DA-SMOTE)
# ═══════════════════════════════════════════════════════════
class SyntheticEmbeddingDataset(Dataset):
    """Holds pseudo-documents whose sentences are DA-SMOTE embeddings."""

    def __init__(self, embeddings_list, labels_list):
        assert len(embeddings_list) == len(labels_list)
        self.embeddings_list = embeddings_list
        self.labels_list     = labels_list

    def __len__(self):
        return len(self.embeddings_list)

    def __getitem__(self, idx):
        emb = torch.tensor(self.embeddings_list[idx], dtype=torch.float32)
        lbl = torch.tensor(self.labels_list[idx],     dtype=torch.long)
        return {"embeddings": emb, "labels": lbl}


def collate_synthetic(batch):
    T_max = max(b["embeddings"].shape[0] for b in batch)
    D, B  = batch[0]["embeddings"].shape[1], len(batch)
    embeddings = torch.zeros(B, T_max, D, dtype=torch.float32)
    labels     = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths    = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["embeddings"].shape[0]
        embeddings[i, :t] = b["embeddings"]
        labels[i, :t]     = b["labels"]
        lengths[i]        = t
    return embeddings, labels, lengths


# ═══════════════════════════════════════════════════════════════════════════
# ███████╗  DISCOURSE-AWARE SMOTE  (DA-SMOTE)  ████████████████████████████
# ═══════════════════════════════════════════════════════════════════════════
class DiscourseAwareSMOTE:
    """
    Discourse-Aware SMOTE for rhetorical-role sequence labelling.

    Three key properties compared to vanilla SMOTE
    ───────────────────────────────────────────────
    1. Intra-class neighbourhood
       Only sentences with the SAME rhetorical label are used as SMOTE
       neighbours.  This preserves label semantics during interpolation.

    2. Positional coherence filter
       Each sentence carries a normalised position p ∈ [0, 1] inside its
       source document (sentence_index / doc_length).  A synthetic sample
       is only generated between two sentences whose positions differ by
       ≤ POSITION_WINDOW.  This enforces the structural regularity of legal
       discourse (e.g. PREAMBLE near the start, RATIO near the end).

    3. Transition-aware pseudo-document assembly
       After generating synthetic embeddings, they are stitched into
       pseudo-documents using a Markov transition matrix estimated from the
       REAL training corpus.  This ensures the context BiLSTM sees plausible
       discourse sequences rather than random permutations.

    Interpolation modes
    ───────────────────
    "embedding":  classic SMOTE linear interpolation in BERT-embedding space.
    "manifold" :  PCA is fitted on each class cluster; interpolation is
                  performed in the reduced PCA space and projected back,
                  keeping synthetics on the data manifold.

    Parameters
    ──────────
    k              : number of same-class neighbours to consider
    position_window: max allowed |p_i - p_j| for a valid interpolation pair
    interp_mode    : "embedding" | "manifold"
    pca_components : PCA dims for manifold mode
    target_ratio   : target_count = target_ratio * majority_count
    min_class_size : minimum real samples to attempt oversampling
    """

    def __init__(
        self,
        k               = DA_K_NEIGHBORS,
        position_window = DA_POSITION_WINDOW,
        interp_mode     = DA_INTERP_MODE,
        pca_components  = DA_PCA_COMPONENTS,
        lambda_low      = DA_LAMBDA_LOW,
        lambda_high     = DA_LAMBDA_HIGH,
        target_ratio    = DA_TARGET_RATIO,
        min_class_size  = DA_MIN_CLASS_SIZE,
        pseudo_doc_size = DA_PSEUDO_DOC_SIZE,
        use_transitions = DA_USE_TRANSITION_CHAIN,
        seed            = SEED,
    ):
        self.k               = k
        self.position_window = position_window
        self.interp_mode     = interp_mode
        self.pca_components  = pca_components
        self.lambda_low      = lambda_low
        self.lambda_high     = lambda_high
        self.target_ratio    = target_ratio
        self.min_class_size  = min_class_size
        self.pseudo_doc_size = pseudo_doc_size
        self.use_transitions = use_transitions
        self.rng             = np.random.default_rng(seed)

    # ── public entry point ────────────────────────────────────────────────
    def fit_resample(
        self,
        embeddings: np.ndarray,   # (N, D) float32
        labels:     np.ndarray,   # (N,)   int64
        positions:  np.ndarray,   # (N,)   float32  ∈ [0, 1]
        doc_ids:    np.ndarray,   # (N,)   int       source-document index
    ):
        """
        Returns
        ───────
        syn_embeddings : list of 2-D arrays  [(T_i, D), ...]  – one per pseudo-doc
        syn_labels     : list of 1-D arrays  [(T_i,),  ...]
        stats          : dict with diagnostics
        """
        class_counts = Counter(labels.tolist())
        majority_cnt = max(class_counts.values())
        target_counts = {
            lbl_id: max(0, int(self.target_ratio * majority_cnt) - class_counts.get(lbl_id, 0))
            for lbl_id in range(NUM_LABELS)
        }

        # Estimate Markov transition matrix from real corpus
        trans_matrix = self._estimate_transition_matrix(labels, doc_ids)

        # Per-class: build PCA model + neighbour graph
        print("\n🔬 Discourse-Aware SMOTE: building per-class neighbour graphs …")
        all_syn_embs  = []
        all_syn_labs  = []
        stats_per_cls = {}

        for lbl_id in range(NUM_LABELS):
            n_needed = target_counts[lbl_id]
            if n_needed <= 0:
                stats_per_cls[id2label[lbl_id]] = {"needed": 0, "generated": 0}
                continue

            mask = labels == lbl_id
            X_cls = embeddings[mask]
            P_cls = positions[mask]      # normalised positions
            n_cls = X_cls.shape[0]

            if n_cls < self.min_class_size:
                print(f"   ⚠ {id2label[lbl_id]:<18}  only {n_cls} samples "
                      f"(< {self.min_class_size}) — skipped")
                stats_per_cls[id2label[lbl_id]] = {
                    "needed": n_needed, "generated": 0, "skipped": True
                }
                continue

            syn_X = self._oversample_class(X_cls, P_cls, n_needed, lbl_id)
            all_syn_embs.extend(syn_X)
            all_syn_labs.extend([lbl_id] * len(syn_X))

            print(f"   ✔ {id2label[lbl_id]:<18}  real={n_cls:4d}  "
                  f"needed={n_needed:4d}  generated={len(syn_X):4d}")
            stats_per_cls[id2label[lbl_id]] = {
                "needed": n_needed, "generated": len(syn_X)
            }

        print(f"\n   Total synthetic sentences : {len(all_syn_labs):,}")

        # Assemble into pseudo-documents
        pseudo_docs_emb, pseudo_docs_lab = self._assemble_pseudo_docs(
            np.array(all_syn_embs, dtype=np.float32),
            np.array(all_syn_labs, dtype=np.int64),
            trans_matrix,
        )
        print(f"   Pseudo-documents created  : {len(pseudo_docs_emb):,}  "
              f"(~{self.pseudo_doc_size} sents each)")

        return pseudo_docs_emb, pseudo_docs_lab, {
            "per_class": stats_per_cls,
            "total_synthetic_sents": len(all_syn_labs),
            "total_pseudo_docs": len(pseudo_docs_emb),
            "transition_matrix": trans_matrix.tolist(),
        }

    # ── per-class oversampling ────────────────────────────────────────────
    def _oversample_class(
        self,
        X:        np.ndarray,   # (n_cls, D)
        P:        np.ndarray,   # (n_cls,)  positions
        n_needed: int,
        lbl_id:   int,
    ):
        """Generate n_needed synthetic embeddings for one class."""
        n_cls = X.shape[0]

        # ── manifold PCA ──────────────────────────────────────────────────
        if self.interp_mode == "manifold":
            n_comp = min(self.pca_components, n_cls - 1, X.shape[1])
            if n_comp < 2:
                # fall back to embedding mode if too few samples
                mode = "embedding"
            else:
                pca  = PCA(n_components=n_comp, random_state=SEED)
                X_r  = pca.fit_transform(X)    # (n_cls, n_comp)
                mode = "manifold"
        else:
            mode = "embedding"

        # ── k-NN in embedding space (for positional filtering) ────────────
        k_eff = min(self.k + 1, n_cls)
        nbrs  = NearestNeighbors(n_neighbors=k_eff, metric="cosine", n_jobs=-1)
        nbrs.fit(X)
        distances, indices = nbrs.kneighbors(X)   # (n_cls, k_eff)
        # indices[:,0] is the point itself; neighbours are indices[:,1:]

        syn_X  = []
        trials = 0
        max_trials = n_needed * 20   # safety cap

        while len(syn_X) < n_needed and trials < max_trials:
            trials += 1
            # pick a random seed sentence
            seed_idx = int(self.rng.integers(0, n_cls))
            p_seed   = P[seed_idx]

            # positional filter: keep neighbours within ±window
            candidate_nbr_idxs = indices[seed_idx, 1:]    # exclude self
            valid_nbrs = [
                nb for nb in candidate_nbr_idxs
                if abs(P[nb] - p_seed) <= self.position_window
            ]
            if not valid_nbrs:
                # relax window if no valid neighbour
                valid_nbrs = list(candidate_nbr_idxs)
            if not valid_nbrs:
                continue

            nbr_idx = int(self.rng.choice(valid_nbrs))
            lam     = float(self.rng.uniform(self.lambda_low, self.lambda_high))

            if mode == "manifold":
                r_seed = X_r[seed_idx]
                r_nbr  = X_r[nbr_idx]
                r_syn  = lam * r_seed + (1.0 - lam) * r_nbr
                e_syn  = pca.inverse_transform(r_syn[np.newaxis])[0]
            else:
                e_syn  = lam * X[seed_idx] + (1.0 - lam) * X[nbr_idx]

            syn_X.append(e_syn.astype(np.float32))

        return syn_X

    # ── Markov transition matrix ──────────────────────────────────────────
    def _estimate_transition_matrix(
        self,
        labels:  np.ndarray,
        doc_ids: np.ndarray,
    ) -> np.ndarray:
        """
        Estimate P(label_j | label_i) from consecutive sentence pairs
        within each document.  Smoothed with add-1 (Laplace).
        """
        trans = np.ones((NUM_LABELS, NUM_LABELS), dtype=np.float64)  # Laplace

        unique_docs = np.unique(doc_ids)
        for d in unique_docs:
            seq = labels[doc_ids == d]
            for i in range(len(seq) - 1):
                trans[seq[i], seq[i + 1]] += 1.0

        # Row-normalise
        row_sums = trans.sum(axis=1, keepdims=True)
        trans    = trans / np.maximum(row_sums, 1e-9)
        return trans.astype(np.float32)

    # ── pseudo-document assembly ──────────────────────────────────────────
    def _assemble_pseudo_docs(
        self,
        syn_embs: np.ndarray,   # (N_syn, D)
        syn_labs: np.ndarray,   # (N_syn,)
        trans:    np.ndarray,   # (C, C)  Markov matrix
    ):
        """
        Pack synthetic sentences into pseudo-documents.

        With DA_USE_TRANSITION_CHAIN=True, the sentences inside each
        pseudo-document are ordered using a greedy Markov walk:
          - start from a randomly chosen sentence
          - at each step pick the next sentence whose label maximises
            trans[current_label, candidate_label], breaking ties randomly.
        This produces pseudo-documents with plausible discourse order.

        With DA_USE_TRANSITION_CHAIN=False, sentences are shuffled randomly.
        """
        N_syn = len(syn_labs)
        if N_syn == 0:
            return [], []

        # shuffle pool
        perm     = self.rng.permutation(N_syn)
        syn_embs = syn_embs[perm]
        syn_labs = syn_labs[perm]

        pseudo_docs_emb, pseudo_docs_lab = [], []

        for start in range(0, N_syn, self.pseudo_doc_size):
            chunk_emb = syn_embs[start : start + self.pseudo_doc_size]
            chunk_lab = syn_labs[start : start + self.pseudo_doc_size]
            if len(chunk_emb) < 2:
                continue

            if self.use_transitions:
                chunk_emb, chunk_lab = self._markov_reorder(
                    chunk_emb, chunk_lab, trans
                )

            pseudo_docs_emb.append(chunk_emb)
            pseudo_docs_lab.append(chunk_lab.tolist())

        return pseudo_docs_emb, pseudo_docs_lab

    def _markov_reorder(
        self,
        embs: np.ndarray,   # (T, D)
        labs: np.ndarray,   # (T,)
        trans: np.ndarray,  # (C, C)
    ):
        """Greedy Markov-walk reordering of a chunk of synthetic sentences."""
        T      = len(labs)
        used   = np.zeros(T, dtype=bool)
        order  = []

        # start from random sentence
        cur = int(self.rng.integers(0, T))
        order.append(cur)
        used[cur] = True

        for _ in range(T - 1):
            cur_lbl = int(labs[cur])
            best_score = -1.0
            best_j     = -1

            for j in range(T):
                if used[j]:
                    continue
                score = float(trans[cur_lbl, int(labs[j])])
                # add tiny random noise to break ties fairly
                score += float(self.rng.uniform(0, 1e-6))
                if score > best_score:
                    best_score = score
                    best_j     = j

            if best_j == -1:
                # all used (shouldn't happen)
                break
            order.append(best_j)
            used[best_j] = True
            cur = best_j

        order_arr = np.array(order, dtype=np.int64)
        return embs[order_arr], labs[order_arr]


# ═══════════════════════════════════════════════════════════════════════════
# EMBEDDING EXTRACTOR  (for DA-SMOTE)
# ═══════════════════════════════════════════════════════════════════════════
def extract_sentence_embeddings_with_positions(
    docs,
    bert_model,
    tokenizer,
    device,
    batch_size=DA_EMBED_BATCH_SIZE,
    max_length=MAX_SEQ_LENGTH,
):
    """
    Returns
    ───────
    embeddings : (N, 768)  mean-pooled BERT sentence embeddings
    labels     : (N,)      integer label ids
    positions  : (N,)      normalised position ∈ [0, 1]
    doc_ids    : (N,)      source document index
    """
    print("\n🔍 Extracting sentence embeddings for DA-SMOTE (frozen BERT) …")
    bert_model.eval()

    flat_sents, flat_labels, flat_positions, flat_doc_ids = [], [], [], []
    for doc_idx, (sents, labs) in enumerate(docs):
        n = len(sents)
        for sent_idx, (sent, lbl) in enumerate(zip(sents, labs)):
            flat_sents.append(sent)
            flat_labels.append(lbl)
            flat_positions.append(sent_idx / max(n - 1, 1))  # ∈ [0,1]
            flat_doc_ids.append(doc_idx)

    all_embeddings = []
    n_batches = (len(flat_sents) + batch_size - 1) // batch_size

    with torch.no_grad():
        for b_idx in range(n_batches):
            batch_sents = flat_sents[b_idx * batch_size: (b_idx + 1) * batch_size]
            enc = tokenizer(
                batch_sents, padding="max_length", truncation=True,
                max_length=max_length, return_tensors="pt",
            )
            input_ids      = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            token_type_ids = enc.get(
                "token_type_ids", torch.zeros_like(enc["input_ids"])
            ).to(device)

            outputs  = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            hidden   = outputs.last_hidden_state              # (B, L, H)
            mask_f   = attention_mask.unsqueeze(-1).float()
            sent_emb = (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)
            all_embeddings.append(sent_emb.cpu().numpy())

            if (b_idx + 1) % 20 == 0:
                print(f"   Batch {b_idx+1}/{n_batches} …", flush=True)

    embeddings = np.vstack(all_embeddings).astype(np.float32)
    labels     = np.array(flat_labels,    dtype=np.int64)
    positions  = np.array(flat_positions, dtype=np.float32)
    doc_ids    = np.array(flat_doc_ids,   dtype=np.int64)

    print(f"   ✔ {embeddings.shape[0]:,} embeddings  "
          f"(dim={embeddings.shape[1]},  {len(docs)} docs)")
    return embeddings, labels, positions, doc_ids


def run_discourse_aware_smote(docs, bert_model, tokenizer, device):
    """
    Full DA-SMOTE pipeline:
      1. Extract sentence embeddings + positions from the training corpus.
      2. Run DiscourseAwareSMOTE.fit_resample().
      3. Return a SyntheticEmbeddingDataset ready for the trainer.
    """
    if not DA_SMOTE_ENABLED:
        print("   DA-SMOTE disabled — training on original data only.")
        return SyntheticEmbeddingDataset([], []), {}

    embeddings, labels, positions, doc_ids = \
        extract_sentence_embeddings_with_positions(
            docs, bert_model, tokenizer, device
        )

    before_counts = Counter(labels.tolist())
    print(f"\n📈 Class distribution BEFORE DA-SMOTE:")
    for lbl_id in range(NUM_LABELS):
        print(f"   {id2label[lbl_id]:<20}  {before_counts.get(lbl_id, 0):5d}")

    da_smote = DiscourseAwareSMOTE()
    pseudo_docs_emb, pseudo_docs_lab, smote_stats = da_smote.fit_resample(
        embeddings, labels, positions, doc_ids
    )

    # Count synthetic sentences
    after_syn = Counter()
    for chunk_lab in pseudo_docs_lab:
        for l in chunk_lab:
            after_syn[l] += 1

    print(f"\n📈 Synthetic sentence distribution (DA-SMOTE):")
    for lbl_id in range(NUM_LABELS):
        syn_cnt  = after_syn.get(lbl_id, 0)
        real_cnt = before_counts.get(lbl_id, 0)
        print(f"   {id2label[lbl_id]:<20}  real={real_cnt:5d}  syn={syn_cnt:4d}  "
              f"total={real_cnt+syn_cnt:5d}")

    return SyntheticEmbeddingDataset(pseudo_docs_emb, pseudo_docs_lab), smote_stats


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query      = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn    = self.attn_drop(F.softmax(attn, dim=-1))
        context = torch.matmul(attn, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2    # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim=sent_out_dim, num_heads=mha_heads, dropout=mha_dropout
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2    # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        n_total = len(self.bert.encoder.layer)
        print(f"\n❄️  BERT frozen : embeddings + layers 0-{n_freeze-1}")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n_total-1} + pooler\n")

    # ── encode a batch of (B, T, L) token tensors → (B, T, sent_out_dim) ─
    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                          lengths=None):
        B, T, L = input_ids.shape
        N       = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid      = flat_mask.sum(-1) > 0

        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)

        token_embs        = self.dropout(token_embs)
        lstm_out, _       = self.sent_bilstm(token_embs)
        lstm_out          = self.dropout(lstm_out)
        pad_mask          = (flat_mask == 0).clone()
        pad_mask[~valid]  = False
        sent_vecs         = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs         = self.sent_layer_norm(sent_vecs)
        sent_vecs         = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    # ── shared CRF head (used by both forward paths) ──────────────────────
    def _ctx_and_crf(self, sent_vecs, labels, lengths):
        sent_vecs = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask    = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss    = -self.crf(emissions, safe_labels, mask=mask,
                                     reduction="mean")
            B2, T2, C   = emissions.shape
            ce_loss     = self.ce_loss(
                emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2)
            )
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions

    # ── standard forward (tokenised text) ─────────────────────────────────
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        return self._ctx_and_crf(sent_vecs, labels, lengths)

    # ── DA-SMOTE forward (pre-computed embeddings) ─────────────────────────
    def forward_from_embeddings(self, embeddings, labels=None, lengths=None):
        """
        Skip BERT + Sentence-BiLSTM + MHA pooling entirely.
        The embeddings are 768-dim BERT mean-pool vectors; they are fed
        directly into the context BiLSTM via a projection layer.

        NOTE: we add a linear projection (bert_dim → sent_out_dim) so that
        the pre-computed 768-d embeddings can enter the context BiLSTM which
        expects sent_out_dim=256 inputs.
        """
        # lazy init of projection layer (avoids adding a dead param at __init__)
        if not hasattr(self, "_emb_proj"):
            sent_out_dim = self.sent_bilstm.hidden_size * 2    # 256
            self._emb_proj = nn.Linear(
                self.bert_dim, sent_out_dim, bias=False
            ).to(embeddings.device)
            nn.init.xavier_uniform_(self._emb_proj.weight)

        proj = self._emb_proj(embeddings)                      # (B, T, 256)
        proj = self.sent_layer_norm(proj)
        return self._ctx_and_crf(proj, labels, lengths)


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    def _f1(av):  return f1_score(all_trues, all_preds, average=av, zero_division=0)
    def _pr(av):  return precision_score(all_trues, all_preds, average=av, zero_division=0)
    def _rc(av):  return recall_score(all_trues, all_preds, average=av, zero_division=0)

    present_rare = [r for r in rare_ids if r in all_trues]
    rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                          average="macro", zero_division=0) if present_rare else 0.0
    rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0) if present_rare else 0.0
    rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                              average="macro", zero_division=0) if present_rare else 0.0

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                               average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                      average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                   average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall": float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }

    return {
        "macro_f1":            _f1("macro"),
        "micro_f1":            _f1("micro"),
        "weighted_f1":         _f1("weighted"),
        "macro_precision":     _pr("macro"),
        "micro_precision":     _pr("micro"),
        "weighted_precision":  _pr("weighted"),
        "macro_recall":        _rc("macro"),
        "micro_recall":        _rc("micro"),
        "weighted_recall":     _rc("weighted"),
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        "per_class_metrics":   per_class_metrics,
        "accuracy":            accuracy_score(all_trues, all_preds),
        "cls_report":          classification_report(str_trues, str_preds,
                                                      labels=LABELS, digits=4,
                                                      zero_division=0),
        "cm":                  confusion_matrix(str_trues, str_preds, labels=LABELS),
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    # ── layer-wise LR decay ───────────────────────────────────────────────
    def build_optimizer(self):
        param_groups = []
        # BERT pooler
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        # BERT encoder layers (top-down decay)
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params,
                    "lr": BERT_LR * (BERT_LR_DECAY ** depth),
                    "weight_decay": WEIGHT_DECAY,
                })
        # Head (BiLSTM, MHA, CRF, classifier, emb_proj)
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        # Also include projection layer if it has been lazily created
        if hasattr(self.model, "_emb_proj"):
            head_params.extend(list(self.model._emb_proj.parameters()))

        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, mask, types, labels, lengths in loader:
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                labels, lengths  = labels.to(self.device), lengths.to(self.device)
                loss, _ = self.model(ids, mask, types, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids, tokenizer,
              num_epochs=NUM_EPOCHS, synthetic_dataset=None):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )

        # ── synthetic loader setup ────────────────────────────────────────
        syn_loader, syn_iter = None, None
        use_synth = (
            synthetic_dataset is not None
            and len(synthetic_dataset) > 0
            and DA_SYNTHETIC_WEIGHT > 0.0
        )
        if use_synth:
            syn_loader = DataLoader(
                synthetic_dataset, batch_size=BATCH_DOCS,
                shuffle=True, collate_fn=collate_synthetic,
            )
            syn_iter = iter(syn_loader)
            print(f"\n🧬 DA-SMOTE synthetic training enabled  "
                  f"({len(synthetic_dataset)} pseudo-docs, "
                  f"weight={DA_SYNTHETIC_WEIGHT})")
        else:
            print("\n   Synthetic training disabled.")

        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None
        total_start   = time.time()
        actual_epochs = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            run_loss, syn_loss_total, n_steps, nan_steps = 0.0, 0.0, 0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, mask, types, labels, lengths) in enumerate(train_loader):
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                labels, lengths  = labels.to(self.device), lengths.to(self.device)

                real_loss, _ = self.model(ids, mask, types, labels=labels, lengths=lengths)

                if torch.isnan(real_loss) or torch.isinf(real_loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                combined = real_loss / GRADIENT_ACCUMULATION_STEPS

                # ── DA-SMOTE synthetic batch ──────────────────────────────
                if use_synth:
                    try:
                        syn_embs, syn_labs, syn_lens = next(syn_iter)
                    except StopIteration:
                        syn_iter = iter(syn_loader)
                        syn_embs, syn_labs, syn_lens = next(syn_iter)

                    syn_embs = syn_embs.to(self.device)
                    syn_labs = syn_labs.to(self.device)
                    syn_lens = syn_lens.to(self.device)

                    syn_loss, _ = self.model.forward_from_embeddings(
                        syn_embs, labels=syn_labs, lengths=syn_lens
                    )
                    if not (torch.isnan(syn_loss) or torch.isinf(syn_loss)):
                        combined         = combined + DA_SYNTHETIC_WEIGHT * syn_loss / GRADIENT_ACCUMULATION_STEPS
                        syn_loss_total  += syn_loss.item()

                combined.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                run_loss += real_loss.item(); n_steps += 1

            # flush remaining gradients
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            avg_loss     = run_loss  / max(1, n_steps)
            avg_syn_loss = syn_loss_total / max(1, n_steps) if use_synth else 0.0
            val_loss     = self.compute_val_loss(dev_dataset)
            val_m        = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan={nan_steps}]" if nan_steps else ""
            syn_info = f" | syn_loss: {avg_syn_loss:.4f}" if use_synth else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_loss:.4f}{syn_info} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                f"val_acc: {val_m['accuracy']:.4f} | "
                f"time: {time.time()-t0:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{nan_info}"
            )

            history.append({
                "epoch": epoch, "train_loss": avg_loss, "syn_loss": avg_syn_loss,
                "val_loss": val_loss,
                "val_accuracy": val_m["accuracy"],
                "val_macro_f1": val_m["macro_f1"],
                "val_micro_f1": val_m["micro_f1"],
                "val_weighted_f1": val_m["weighted_f1"],
                "val_rare_f1": val_m["rare_f1"],
                "val_macro_precision": val_m["macro_precision"],
                "val_micro_precision": val_m["micro_precision"],
                "val_weighted_precision": val_m["weighted_precision"],
                "val_rare_precision": val_m["rare_precision"],
                "val_macro_recall": val_m["macro_recall"],
                "val_micro_recall": val_m["micro_recall"],
                "val_weighted_recall": val_m["weighted_recall"],
                "val_rare_recall": val_m["rare_recall"],
                "epoch_train_time_s": time.time() - t0,
                "nan_steps": nan_steps,
                "timestamp": datetime.utcnow().isoformat(),
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  Early stopping after epoch {epoch} "
                      f"(patience={early_stopper.patience}).\n")
                break

        total_time = time.time() - total_start
        print(f"\n⏱  Total training : {total_time/60:.2f} min  "
              f"({actual_epochs} epochs)")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing = {
            "total_training_time_s":   total_time,
            "total_training_time_min": total_time / 60,
            "avg_epoch_time_s":        total_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time()

        with torch.no_grad():
            for ids, mask, types, labels, lengths in loader:
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, mask, types, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].cpu().numpy().tolist())

        if measure_inference_time:
            elapsed      = time.time() - t0
            n_sents      = len(all_trues)
            infer_info   = {
                "split": split_name, "n_documents": n_samples,
                "n_sentences": n_sents,
                "total_inference_time_s": elapsed,
                "latency_per_document_ms": elapsed / max(1, n_samples) * 1000,
                "latency_per_sentence_ms": elapsed / max(1, n_sents) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, elapsed),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {elapsed:.2f}s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f}ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name": INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden": CTX_LSTM_HIDDEN,
            "mha_heads": MHA_HEADS, "mha_dropout": MHA_DROPOUT,
            "num_labels": NUM_LABELS, "dropout": DROPOUT,
            "labels": LABELS, "label2id": label2id, "id2label": id2label,
            "max_seq_length": MAX_SEQ_LENGTH,
            "freeze_layers": BERT_FREEZE_LAYERS,
            "da_smote": {
                "enabled": DA_SMOTE_ENABLED,
                "k": DA_K_NEIGHBORS,
                "position_window": DA_POSITION_WINDOW,
                "interp_mode": DA_INTERP_MODE,
                "pca_components": DA_PCA_COMPONENTS,
                "target_ratio": DA_TARGET_RATIO,
                "pseudo_doc_size": DA_PSEUDO_DOC_SIZE,
                "synthetic_weight": DA_SYNTHETIC_WEIGHT,
                "use_transitions": DA_USE_TRANSITION_CHAIN,
            },
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss",  marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",    marker="s")
        if "syn_loss" in hist_df and hist_df["syn_loss"].any():
            ax.plot(epochs, hist_df["syn_loss"], label="Syn Loss (DA-SMOTE)",
                    marker="^", linestyle="--")
        ax.set_title("Training vs Validation Loss  (DA-SMOTE)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1  (DA-SMOTE)")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].plot(epochs, hist_df["train_loss"], label="Train", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision", "Macro-Prec", "-"),
            ("val_micro_precision", "Micro-Prec", "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision", "Rare-Prec", ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall", "Macro-Rec", "-"),
            ("val_micro_recall", "Micro-Rec", "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall", "Rare-Rec", ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(hist_df["epoch_train_time_s"].mean(), color="red", linestyle="--",
                       label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s")
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y"); plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS, cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix  (DA-SMOTE)"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1  (DA-SMOTE)"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 70)
    print("FINAL RESULTS  (InLegalBERT+BiLSTM+MHA+CRF  +  DA-SMOTE)")
    print("=" * 70)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 70)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 70)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 70)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 70)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4 layers) → Sentence BiLSTM(128) "
          "→ MHA(4-head) → Context BiLSTM(64) → Linear → CRF + Aux-CE")
    print("\nAnti-overfitting settings:")
    print(f"  Dropout={DROPOUT}  WD={WEIGHT_DECAY}  "
          f"Freeze={BERT_FREEZE_LAYERS} layers  LR-decay={BERT_LR_DECAY}")
    print(f"  Aux CE weight={AUX_CE_WEIGHT}  label_smoothing={LABEL_SMOOTHING}")
    print(f"  Early stopping patience={ES_PATIENCE}")
    print("\nDA-SMOTE settings:")
    print(f"  Enabled         : {DA_SMOTE_ENABLED}")
    print(f"  Interp mode     : {DA_INTERP_MODE}  (PCA dims={DA_PCA_COMPONENTS})")
    print(f"  k neighbours    : {DA_K_NEIGHBORS}")
    print(f"  Position window : ±{DA_POSITION_WINDOW*100:.0f}% of doc length")
    print(f"  Target ratio    : {DA_TARGET_RATIO*100:.0f}% of majority class")
    print(f"  λ range         : [{DA_LAMBDA_LOW}, {DA_LAMBDA_HIGH}]")
    print(f"  Transition chain: {DA_USE_TRANSITION_CHAIN}")
    print(f"  Synthetic weight: {DA_SYNTHETIC_WEIGHT}\n")

    print("Loading JSONL files …")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer …")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    print("\nInitialising model …")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    # ── DA-SMOTE augmentation ─────────────────────────────────────────────
    print("\n" + "═" * 65)
    print("  DISCOURSE-AWARE SMOTE  AUGMENTATION")
    print("═" * 65)
    model.to(DEVICE)
    synthetic_dataset, smote_stats = run_discourse_aware_smote(
        train_docs, model.bert, tokenizer, DEVICE
    )
    if smote_stats:
        with open(os.path.join(OUT_DIR, "da_smote_stats.json"), "w") as f:
            json.dump(smote_stats, f, indent=2, default=str)
    print("═" * 65 + "\n")

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training (max {NUM_EPOCHS} epochs, "
          f"early-stop patience={ES_PATIENCE}) …")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids          = rare_ids,
        tokenizer         = tokenizer,
        num_epochs        = NUM_EPOCHS,
        synthetic_dataset = synthetic_dataset,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev …")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy={dev_metrics['accuracy']:.4f}  "
          f"Macro-F1={dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1={dev_metrics['rare_f1']:.4f}")
    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF  +  DA-SMOTE\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])
    trainer.save_confusion_matrix(dev_metrics["cm"],           "dev",  rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev",  rare_labels)

    print("\nEvaluating on Test …")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy={test_metrics['accuracy']:.4f}  "
          f"Macro-F1={test_metrics['macro_f1']:.4f}  "
          f"Rare-F1={test_metrics['rare_f1']:.4f}")
    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF  +  DA-SMOTE\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])
    trainer.save_confusion_matrix(test_metrics["cm"],            "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name": "InLegalBERT+BiLSTM+MHA+CRF  +  DA-SMOTE",
            "bert_model": INLEGALBERT_MODEL_NAME,
            "trainable_params": total_trainable,
            "frozen_params": total_frozen,
        },
        "da_smote": {
            "enabled": DA_SMOTE_ENABLED,
            "interp_mode": DA_INTERP_MODE,
            "k": DA_K_NEIGHBORS,
            "position_window": DA_POSITION_WINDOW,
            "target_ratio": DA_TARGET_RATIO,
            "synthetic_weight": DA_SYNTHETIC_WEIGHT,
            "use_transitions": DA_USE_TRANSITION_CHAIN,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(dev_metrics, test_metrics,
                        total_train_time=total_train_time,
                        total_trainable=total_trainable,
                        total_frozen=total_frozen)

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-4 layers) → Sentence BiLSTM(128) → MHA(4-head) → Context BiLSTM(64) → Linear → CRF + Aux-CE

Anti-overfitting settings:
  Dropout=0.4  WD=0.05  Freeze=8 layers  LR-decay=0.9
  Aux CE weight=0.2  label_smoothing=0.1
  Early stopping patience=10

DA-SMOTE settings:
  Enabled         : True
  Interp mode     : manifold  (PCA dims=32)
  k neighbours    : 5
  Position window : ±15% of doc length
  Target ratio    : 60% of majority class
  λ range         : [0.3, 0.7]
  Transition chain: True
  Synthetic weight: 0.4

Loading JSONL files …
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS      

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen : embeddings + layers 0-7
🔥 BERT trainable: layers 8-11 + pooler


MODEL PARAMETER SUMMARY  (InLegalBERT+BiLSTM+MHA+CRF  DA-SMOTE)
  Component                           Trainable     Frozen        Total
------------------------------------------------------------------------
  InLegalBERT Encoder                28,942,080 80,540,160  109,482,240
  Sentence BiLSTM                     1,314,816          0    1,314,816
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        264,192          0      264,192
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
────────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        30,728,016 80,540,160  111,268,176

═════════════════════════════════════════════════════════════════
  DISCOURSE-AWARE SMOTE  AUGMENTATION
═════════════════════════